# Celda 0 — Configuración

In [ ]:
# ════════════════════════════════════
# CELDA 0 — Configuración: Drive y rutas
# ════════════════════════════════════
import os

try:
    # Entorno Google Colab: se monta Drive (como pide el PDF)
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
    ENTORNO = 'Colab + Drive'
except ImportError:
    # Entorno local (VS Code): carpeta datos/ junto al notebook
    RUTA_BASE = os.path.join(os.getcwd(), 'datos', '')
    ENTORNO = 'Local (VS Code)'

os.makedirs(RUTA_BASE, exist_ok=True)

# Rutas de cada archivo de persistencia
RUTA_USUARIOS   = RUTA_BASE + 'usuarios.txt'
RUTA_LIBROS     = RUTA_BASE + 'libros.txt'
RUTA_CATEGORIAS = RUTA_BASE + 'categorias.txt'
RUTA_PRESTAMOS  = RUTA_BASE + 'prestamos.txt'
RUTA_MULTAS     = RUTA_BASE + 'multas.txt'
RUTA_RESERVAS   = RUTA_BASE + 'reservas.txt'

if os.path.isdir(RUTA_BASE):
    print(f'✔ Entorno: {ENTORNO}')
    print('✔ Carpeta de datos lista en:', RUTA_BASE)
else:
    print('✘ No se pudo crear la carpeta de datos.')

    

# Sección 1 — Modelo: Enums

In [2]:
# ════════════════════════════════════
# CELDA 1.1 — Enums
# ════════════════════════════════════
from enum import Enum


class TipoUsuarioEnum(Enum):
    """Tipos de usuario de la biblioteca."""
    ESTUDIANTE = 'ESTUDIANTE'
    DOCENTE = 'DOCENTE'
    EXTERNO = 'EXTERNO'


class EstadoPrestamoEnum(Enum):
    """Estados posibles de un préstamo."""
    ACTIVO = 'ACTIVO'
    DEVUELTO = 'DEVUELTO'
    VENCIDO = 'VENCIDO'


class EstadoMultaEnum(Enum):
    """Estados posibles de una multa."""
    PENDIENTE = 'PENDIENTE'
    PAGADA = 'PAGADA'


class EstadoReservaEnum(Enum):
    """Estados posibles de una reserva."""
    ACTIVA = 'ACTIVA'
    CUMPLIDA = 'CUMPLIDA'
    CANCELADA = 'CANCELADA'

# Sección 2 — Repositorios: ArchivoUtil

In [3]:
# ════════════════════════════════════
# CELDA 2.1 — ArchivoUtil
# ════════════════════════════════════
import os
from datetime import datetime


class ArchivoUtil:
    """Utilidades para leer y escribir archivos de texto (.txt) en Google Drive.

    Cada línea del archivo es un registro. Todos los métodos son estáticos,
    por lo que no hace falta crear un objeto para usarlos.
    """

    @staticmethod
    def existe(ruta):
        """Retorna True si el archivo existe en la ruta indicada."""
        return os.path.isfile(ruta)

    @staticmethod
    def crear_si_no_existe(ruta):
        """Crea un archivo vacío si no existe.

        Retorna True si lo creó y False si ya existía.
        """
        if os.path.isfile(ruta):
            return False
        carpeta = os.path.dirname(ruta)
        if carpeta:
            os.makedirs(carpeta, exist_ok=True)
        with open(ruta, 'a', encoding='utf-8'):
            pass
        return True

    @staticmethod
    def leer_lineas(ruta):
        """Lee todas las líneas no vacías de un archivo.

        Retorna una lista vacía si el archivo no existe.
        """
        try:
            with open(ruta, 'r', encoding='utf-8') as f:
                return [l.strip() for l in f if l.strip()]
        except FileNotFoundError:
            return []

    @staticmethod
    def escribir_linea(ruta, linea):
        """Agrega una línea al final del archivo.

        Lanza ValueError si la línea contiene saltos de línea, porque
        rompería la regla de un registro por línea.
        """
        if '\n' in linea or '\r' in linea:
            raise ValueError('La línea no puede contener saltos de línea.')
        with open(ruta, 'a', encoding='utf-8') as f:
            f.write(linea + '\n')

    @staticmethod
    def reescribir(ruta, lineas):
        """Reemplaza el contenido completo del archivo con las líneas dadas.

        Si la lista está vacía, deja el archivo vacío.
        """
        with open(ruta, 'w', encoding='utf-8') as f:
            if lineas:
                f.write('\n'.join(lineas) + '\n')

    @staticmethod
    def generar_codigo(prefijo):
        """Genera un código único basado en la fecha y hora actuales.

        Ejemplo: generar_codigo('PRE') -> 'PRE-20251015083000123'
        """
        ts = datetime.now().strftime('%Y%m%d%H%M%S%f')[:17]
        return f'{prefijo}-{ts}'

# Verificación P0


In [4]:
# ════════════════════════════════════
# VERIFICACIÓN P0 — Persistencia en Drive
# ════════════════════════════════════
import re

# 1) Crear los 6 archivos si no existen
rutas = {
    'usuarios.txt': RUTA_USUARIOS,
    'categorias.txt': RUTA_CATEGORIAS,
    'libros.txt': RUTA_LIBROS,
    'prestamos.txt': RUTA_PRESTAMOS,
    'multas.txt': RUTA_MULTAS,
    'reservas.txt': RUTA_RESERVAS,
}
print('--- Archivos de persistencia ---')
for nombre, ruta in rutas.items():
    creado = ArchivoUtil.crear_si_no_existe(ruta)
    estado = 'creado' if creado else 'ya existía'
    print(f'  {nombre:<15} {estado:<11} existe={ArchivoUtil.existe(ruta)}')

# 2) Prueba de escritura y lectura con un archivo temporal
RUTA_PRUEBA = RUTA_BASE + 'prueba_p0.txt'
if ArchivoUtil.existe(RUTA_PRUEBA):
    os.remove(RUTA_PRUEBA)

print('\n--- Prueba de lectura/escritura ---')
assert ArchivoUtil.leer_lineas(RUTA_PRUEBA) == [], 'Archivo inexistente debe dar []'
print('  ✔ Leer archivo inexistente devuelve lista vacía')

ArchivoUtil.escribir_linea(RUTA_PRUEBA, '1098765432|Maria Lopez|ESTUDIANTE|maria.lopez@correo.com|3001234567|ACTIVO')
ArchivoUtil.escribir_linea(RUTA_PRUEBA, '1001234567|Juan Perez|DOCENTE|juan.perez@correo.com|3109876543|ACTIVO')
lineas = ArchivoUtil.leer_lineas(RUTA_PRUEBA)
assert len(lineas) == 2, 'Debían leerse 2 líneas'
assert lineas[0].split('|')[1] == 'Maria Lopez'
print('  ✔ escribir_linea + leer_lineas:', len(lineas), 'registros')

ArchivoUtil.reescribir(RUTA_PRUEBA, [lineas[1]])
assert ArchivoUtil.leer_lineas(RUTA_PRUEBA) == [lineas[1]]
print('  ✔ reescribir: el archivo quedó con 1 registro')

codigo = ArchivoUtil.generar_codigo('PRE')
assert re.fullmatch(r'PRE-\d{17}', codigo), f'Formato inesperado: {codigo}'
print('  ✔ generar_codigo:', codigo)

try:
    ArchivoUtil.escribir_linea(RUTA_PRUEBA, 'línea\ncon salto')
    print('  ✘ Debió lanzar ValueError')
except ValueError:
    print('  ✔ Línea con salto de línea rechazada (ValueError)')

os.remove(RUTA_PRUEBA)
print('\n✔ P0 verificado. Archivos en la carpeta:', sorted(os.listdir(RUTA_BASE)))

--- Archivos de persistencia ---
  usuarios.txt    creado      existe=True
  categorias.txt  creado      existe=True
  libros.txt      creado      existe=True
  prestamos.txt   creado      existe=True
  multas.txt      creado      existe=True
  reservas.txt    creado      existe=True

--- Prueba de lectura/escritura ---
  ✔ Leer archivo inexistente devuelve lista vacía
  ✔ escribir_linea + leer_lineas: 2 registros
  ✔ reescribir: el archivo quedó con 1 registro
  ✔ generar_codigo: PRE-20261026151419685
  ✔ Línea con salto de línea rechazada (ValueError)

✔ P0 verificado. Archivos en la carpeta: ['categorias.txt', 'libros.txt', 'multas.txt', 'prestamos.txt', 'reservas.txt', 'usuarios.txt']
